In [ ]:
import requests
import csv
import datetime
import time

In [ ]:

def get_all_data(year, year2):

    team_codes = {
        1 : "NJD", 2 : "NYI", 3 : "NYR",
        4 : "PHI", 5 : "PIT", 6 : "BOS",
        7 : "BUF", 8 : "MTL", 9 : "OTT",
        10 : "TOR", 12 : "CAR", 13 : "FLA",
        14 : "TBL", 15 : "WSH", 16 : "CHI",
        17 : "DET", 18 : "NSH", 19 : "STL",
        20 : "CGY", 21 : "COL", 22 : "EDM",
        23 : "VAN", 24 : "ANA", 25 : "DAL",
        26 : "LAK", 27 : "PHX", 28 : "SJS",
        29 : "CBJ", 30 : "MIN", 52 : "WPG",
        53 : "ARI", 54 : "VGK", 55 : "SEA",
        59 : "UTA"

    }

    game_count = 1

    team_data = []
    player_data = []

    start_date = datetime.date(year, 10, 1)
    end_date = datetime.date(year2, 4, 15)
    current_date = start_date

    teams_data_file = current_date.strftime("%Y") + '_' + end_date.strftime("%Y") + '_All_Teams_Data_TEST.csv'
    players_data_file = current_date.strftime("%Y") + '_' + end_date.strftime("%Y") + '_All_Players_Data_TEST.csv'

    team_data.append(["Team_Name", "Season", "Games_Played", "Wins", "Losses", "x_Goals_For", "x_Goals_Against", "x_SOG_For", "x_SOG_Against"])
    player_data.append(["Player_ID", "Player_Name", "Season", "Current_Team", "Games_Played", "x_Goals", "x_Assists", "x_Shots", "x_Blocked_Shots", "x_Penalty_Mins", "Faceoff_Wins", "Total_Faceoffs"])

    #"x_Rebound_Percentage_For", "x_Rebound_Percentage_Against", "x_Blocked_Percentage_For", "x_Blocked_Percentage_Against","x_Missed_Shots_For", "x_Missed_Shots_For"]
    while True:
        if game_count % 100 == 0:
            print(f"Processed {game_count} games so far...")


        game_id = start_date.strftime("%Y") + "02" + f"{game_count:04}"
        url = f'https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play'

        res = requests.get(url)
        if res.status_code == 404:
            print("Error 404: Not Found")
            break

        data = res.json()

        away = data.get('awayTeam', [])
        home = data.get('homeTeam', [])

        # --------------------------- BASIC DATA ---------------------------------------
        home_win = True
        if away["score"] > home["score"]:
            home_win = False
        if away["abbrev"] not in [team[0] for team in team_data[1:]]:
            team_data.append([away["abbrev"], current_date.strftime("%Y"), 1, 1 if not home_win else 0, 1 if home_win else 0,
                            away["score"], home["score"], away["sog"], home["sog"]])
        else:
            #"Team_Name", "Season", "Games_Played", "Wins", "Losses", "x_Goals_For", "x_Goals_Against", "x_SOG_For", "x_SOG_Against"
            index = next((i for i, team in enumerate(team_data[1:], start=1) if team[0] == away["abbrev"]), None)
            team_data[index][2] += 1
            if not home_win:
                team_data[index][3] += 1
            else:
                team_data[index][4] += 1
            team_data[index][5] = int((team_data[index][5] * (team_data[index][2] - 1))) + int(away["score"]) / team_data[index][2]
            team_data[index][6] = int((team_data[index][6] * (team_data[index][2] - 1))) + int(home["score"]) / team_data[index][2]
            team_data[index][7] = int((team_data[index][7] * (team_data[index][2] - 1))) + int(away["sog"]) / team_data[index][2]
            team_data[index][8] = int((team_data[index][8] * (team_data[index][2] - 1))) + int(home["sog"]) / team_data[index][2]

        if home["abbrev"] not in [team[0] for team in team_data[1:]]:
            team_data.append([home["abbrev"], current_date.strftime("%Y"), 1, 0 if away["score"] > home["score"] else 1, 1 if away["score"] > home["score"] else 0,
                            home["score"], away["score"], home["sog"], away["sog"]])
        else:
            #"Team_Name", "Season", "Games_Played", "Wins", "Losses", "x_Goals_For", "x_Goals_Against", "x_SOG_For", "x_SOG_Against"
            index = next((i for i, team in enumerate(team_data[1:], start=1) if team[0] == home["abbrev"]), None)
            team_data[index][2] += 1
            if home_win:
                team_data[index][3] += 1
            else:
                team_data[index][4] += 1
            team_data[index][5] = int((team_data[index][5] * (team_data[index][2] - 1))) + int(home["score"]) / team_data[index][2]
            team_data[index][6] = int((team_data[index][6] * (team_data[index][2] - 1))) + int(away["score"]) / team_data[index][2]
            team_data[index][7] = int((team_data[index][7] * (team_data[index][2] - 1))) + int(home["sog"]) / team_data[index][2]
            team_data[index][8] = int((team_data[index][8] * (team_data[index][2] - 1))) + int(away["sog"]) / team_data[index][2]

        # --------------------------- ADVANCED DATA ---------------------------------------
        #"Player_ID", "Player_Name", "Season", "Current_Team", "Games Played", "x_Goals", "x_Assists", "x_Shots", "x_Blocked_Shots", "x_Penalty_Mins", "Faceoff_Percentage, "Total Faceoffs",
        for player in data.get("rosterSpots", []):
            if player["playerId"] not in [player[0] for player in player_data[1:]]:
                if player["positionCode"] != "G":
                    player_data.append([player["playerId"], player["firstName"]["default"] + player["lastName"]["default"], current_date.strftime("%Y"),
                                       team_codes[player["teamId"]], 1, 0, 0, 0, 0, 0, 0, 0])
            else:
                p_index = next((i for i, p in enumerate(player_data[1:], start=1) if p[0] == player["playerId"]), None)
                player_data[p_index][4] += 1
                player_data[p_index][3] = team_codes[player["teamId"]]

        for event in data.get("plays", []):
           match event["typeCode"]:
                case 502:
                    l_index = next((i for i, p in enumerate(player_data[1:], start=1) if p[0] == event["details"]["losingPlayerId"]), None)
                    w_index = next((i for i, p in enumerate(player_data[1:], start=1) if p[0] == event["details"]["winningPlayerId"]), None)

                    player_data[w_index][11] += 1
                    player_data[w_index][10] = max(0, (float((player_data[w_index][10] * (player_data[w_index][11] - 1))) + 1.0) / player_data[w_index][11])

                    player_data[l_index][11] += 1
                    player_data[l_index][10] = max(0, (float((player_data[l_index][10] * (player_data[l_index][11] - 1))) - 1.0) / player_data[l_index][11])

                case 509:
                   committed_player_id = event["details"].get("committedByPlayerId")
                   if committed_player_id is not None:
                        p_index = next((i for i, player in enumerate(player_data[1:], start=1) if player[0] == committed_player_id), None)
                        if p_index is not None:
                            player_data[p_index][9] = (float((player_data[p_index][9] * (player_data[p_index][4] - 1))) + event["details"]["duration"]) / player_data[p_index][4]


                case 506:
                   shooting_player_id = event["details"].get("shootingPlayerId")
                   if shooting_player_id is not None:
                    p_index = next((i for i, player in enumerate(player_data[1:], start=1) if player[0] == shooting_player_id), None)
                    if p_index is not None:
                        player_data[p_index][7] = (int((player_data[p_index][7] * (player_data[p_index][4] - 1))) + 1) / player_data[p_index][4]

                case 508:
                   blocking_player_id = event["details"].get("blockingPlayerId")
                   if blocking_player_id is not None:
                        p_index = next((i for i, player in enumerate(player_data[1:], start=1) if player[0] == blocking_player_id), None)
                        if p_index is not None:
                            player_data[p_index][8] = (int((player_data[p_index][8] * (player_data[p_index][4] - 1))) + 1) / player_data[p_index][4]

                case 505:
                   gs_player_id = event["details"].get("scoringPlayerId")
                   a1_player_id = event["details"].get("assist1PlayerId")
                   a2_player_id = event["details"].get("assist2PlayerId")
                   if a1_player_id is not None:
                    pa_index = next((i for i, player in enumerate(player_data[1:], start=1) if player[0] == a1_player_id), None)
                    if pa_index is not None:
                                player_data[pa_index][6] = (int((player_data[pa_index][6] * (player_data[pa_index][4] - 1))) + 1) / player_data[pa_index][4]
                   if a2_player_id is not None:
                    sa_index = next((i for i, player in enumerate(player_data[1:], start=1) if player[0] == a2_player_id), None)
                    if sa_index is not None:
                                player_data[sa_index][6] = (int((player_data[sa_index][6] * (player_data[sa_index][4] - 1))) + 1) / player_data[sa_index][4]
                   if gs_player_id is not None:
                    gs_index = next((i for i, player in enumerate(player_data[1:], start=1) if player[0] == gs_player_id), None)
                    if gs_index is not None:
                        player_data[gs_index][5] = (int((player_data[gs_index][5] * (player_data[gs_index][4] - 1))) + 1) / player_data[gs_index][4]



                case _:
                   pass

        current_date += datetime.timedelta(days=1)

        game_count += 1

    with open(teams_data_file, 'w', newline='') as file:
        csvwriter = csv.writer(file)
        csvwriter.writerows([[f"{col:.2f}" if isinstance(col, float) else col for col in row] for row in team_data])
    with open(players_data_file, 'w', newline='') as file:
        csvwriter = csv.writer(file)
        csvwriter.writerows([[f"{col:.2f}" if isinstance(col, float) else col for col in row] for row in player_data])
    print(f"CSV file '{players_data_file}' created successfully.")

#for i in range(2015, 2025):
get_all_data(2023, 2024)
